# Step 1a: Download podcast audio with yt-dlp

Downloads the audio of every YouTube link in `data/podcasts.md` and saves it as WAV in `data/raw_audios/`.

- Needs `ffmpeg` on your PATH.
- Links that were already downloaded are skipped, so you can run it again after failures.
- If yt-dlp keeps failing, try `01b_download_audio_pytubefix.ipynb` instead.
- Already have the audio, or only a video file? Skip to step 2 or 3.

In [ ]:
import re
import shutil
from pathlib import Path

from yt_dlp import YoutubeDL

# Find the project folder (works from any notebook folder)
project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file() and (path / "src").is_dir()
)
podcasts_file = project_root / "data" / "podcasts.md"
output_dir = project_root / "data" / "raw_audios"
output_dir.mkdir(parents=True, exist_ok=True)

## 1. Read the links

In [ ]:
links = re.findall(r"https?://[^\s)>]+", podcasts_file.read_text(encoding="utf-8"))
print(f"Found {len(links)} links in {podcasts_file.relative_to(project_root)}")
for link in links:
    print(" ", link)

## 2. Download

Leave `cookies_from_browser` as `None` first. If YouTube asks you to sign in, sign in to YouTube in your browser and set it to `"chrome"`, `"firefox"` or `"safari"`. YouTube can still reject the request.

In [ ]:
cookies_from_browser = None  # e.g. "chrome", "firefox" or "safari"


def download_audio(link):
    options = {
        "format": "bestaudio/best",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
        "outtmpl": str(output_dir / "%(title)s.%(ext)s"),
        "noplaylist": True,
        "remote_components": ["ejs:github"],
        # Remembers finished downloads so a second run skips them
        "download_archive": str(output_dir / ".ytdlp_downloaded.txt"),
    }
    node_path = shutil.which("node")
    if node_path:
        options["js_runtimes"] = {"node": {"path": node_path}}
    if cookies_from_browser:
        options["cookiesfrombrowser"] = (cookies_from_browser,)

    with YoutubeDL(options) as ydl:
        ydl.download([link])


failed = []
for index, link in enumerate(links, start=1):
    print(f"({index}/{len(links)}) {link}")
    try:
        download_audio(link)
    except Exception as error:
        failed.append(link)
        print(f"Failed: {error}")

print(f"Finished: {len(links) - len(failed)} succeeded, {len(failed)} failed.")
print(f"Audio folder: {output_dir.relative_to(project_root)}")